In [ ]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

# FD Grid-Independence Study

This notebook runs the steady-state finite-difference solver on grids of 250, 500 and 1000 intervals while keeping the device thickness fixed at 500 nm, and compares each solution with the finest grid (Section 2.3 of the paper). The grid size is passed to `FD_Solver.ipynb` through the variable `L`.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------------------------------------------------------
# Working directory: change only if your project folder is different
# -----------------------------------------------------------------------------
WORKDIR = os.getcwd()  # repository root
os.chdir(WORKDIR)
print("Working directory:", os.getcwd())

# -----------------------------------------------------------------------------
# FD notebook and baseline physical parameters
# -----------------------------------------------------------------------------
FD_NOTEBOOK = "FD_Solver.ipynb"

BASE_P0 = 1.0e21
BASE_PHI = 0.10
BASE_VI = 0.001

# Number of spatial intervals. Device thickness remains fixed at 500 nm.
GRID_INTERVALS = [250, 500, 1000]

OUTPUT_ROOT = os.path.join("results/fd_grid_independence")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Grid intervals:", GRID_INTERVALS)
print("Physical spacings (nm):", [500.0 / L for L in GRID_INTERVALS])

In [ ]:
# -----------------------------------------------------------------------------
# Run the FD solver for each grid
# -----------------------------------------------------------------------------
run_rows = []

for L_value in GRID_INTERVALS:
    dx_nm = 500.0 / L_value
    tag = f"L_{L_value}_dx_{dx_nm:g}nm"
    outdir = os.path.join(OUTPUT_ROOT, tag)
    os.makedirs(outdir, exist_ok=True)

    # Pass parameters into the FD notebook. The notebook must use globals().get(...)
    # for L, P0, phia, phic, Vi, and OUTPUT_DIR.
    g = globals()
    g["L"] = int(L_value)
    g["P0"] = float(BASE_P0)
    g["phia"] = float(BASE_PHI)
    g["phic"] = float(BASE_PHI)
    g["Vi"] = float(BASE_VI)
    g["OUTPUT_DIR"] = outdir

    print("\n" + "=" * 80)
    print(f"Running FD grid: {L_value} intervals")
    print(f"Physical spacing: {dx_nm:.3f} nm")
    print(f"Output directory: {outdir}")
    print("=" * 80)

    t0 = time.time()
    get_ipython().run_line_magic("run", f'-i "{FD_NOTEBOOK}"')
    runtime = time.time() - t0

    required = ["fd_psi.csv", "fd_n.csv", "fd_I.csv"]
    missing = [f for f in required if not os.path.exists(os.path.join(outdir, f))]
    if missing:
        raise FileNotFoundError(f"Missing FD outputs for L={L_value}: {missing}")

    run_rows.append({
        "intervals": int(L_value),
        "nodes": int(L_value + 1),
        "dx_nm": float(dx_nm),
        "runtime_s": float(runtime),
        "output_dir": outdir,
    })

run_df = pd.DataFrame(run_rows)
run_df.to_csv(os.path.join(OUTPUT_ROOT, "grid_runs.csv"), index=False)
display(run_df)

In [ ]:
# -----------------------------------------------------------------------------
# Helper functions for grid comparison
# -----------------------------------------------------------------------------
def load_field(base_dir, field):
    path = os.path.join(base_dir, f"fd_{field}.csv")
    df = pd.read_csv(path)
    if field == "psi":
        value_col = "psi_V"
    elif field == "n":
        value_col = "n_m3"
    elif field == "I":
        value_col = "I_m3"
    else:
        raise ValueError(field)
    return df["x_nm"].to_numpy(float), df[value_col].to_numpy(float)

def rel_l2(candidate, reference):
    return np.linalg.norm(candidate - reference) / np.linalg.norm(reference)

def compare_to_finest(candidate_dir, reference_dir, field):
    x_c, y_c = load_field(candidate_dir, field)
    x_r, y_r = load_field(reference_dir, field)

    # Restrict the reference grid to the common spatial range, then interpolate
    # the candidate solution onto those reference points.
    mask = (x_r >= x_c.min()) & (x_r <= x_c.max())
    x_common = x_r[mask]
    y_ref = y_r[mask]
    y_c_interp = np.interp(x_common, x_c, y_c)

    return {
        "rel_l2": rel_l2(y_c_interp, y_ref),
        "max_abs": float(np.max(np.abs(y_c_interp - y_ref))),
        "x_common": x_common,
        "candidate_interp": y_c_interp,
        "reference": y_ref,
    }

In [ ]:
# -----------------------------------------------------------------------------
# Compute relative differences using the finest grid as the reference
# -----------------------------------------------------------------------------
L_ref = max(GRID_INTERVALS)
ref_row = run_df.loc[run_df["intervals"] == L_ref].iloc[0]
ref_dir = ref_row["output_dir"]

summary_rows = []

for _, row in run_df.iterrows():
    L_value = int(row["intervals"])
    if L_value == L_ref:
        summary_rows.append({
            "intervals": L_value,
            "nodes": int(row["nodes"]),
            "dx_nm": float(row["dx_nm"]),
            "relL2_psi_vs_finest": 0.0,
            "relL2_n_vs_finest": 0.0,
            "relL2_I_vs_finest": 0.0,
        })
        continue

    outdir = row["output_dir"]
    cpsi = compare_to_finest(outdir, ref_dir, "psi")
    cn = compare_to_finest(outdir, ref_dir, "n")
    cI = compare_to_finest(outdir, ref_dir, "I")

    summary_rows.append({
        "intervals": L_value,
        "nodes": int(row["nodes"]),
        "dx_nm": float(row["dx_nm"]),
        "relL2_psi_vs_finest": cpsi["rel_l2"],
        "relL2_n_vs_finest": cn["rel_l2"],
        "relL2_I_vs_finest": cI["rel_l2"],
    })

summary_df = pd.DataFrame(summary_rows).sort_values("intervals")
summary_path = os.path.join(OUTPUT_ROOT, "grid_independence_summary.csv")
summary_df.to_csv(summary_path, index=False)

print(f"Finest-grid reference: {L_ref} intervals ({500.0/L_ref:.3f} nm)")
display(summary_df)
print("Saved:", summary_path)

In [ ]:
# -----------------------------------------------------------------------------
# Overlay plots for visual grid-independence check
# -----------------------------------------------------------------------------
for field, ylabel in [("psi", "Potential (V)"), ("n", r"Electron density (m$^{-3}$)"), ("I", r"Mobile-ion density (m$^{-3}$)")]:
    plt.figure(figsize=(6.2, 4.2))
    for _, row in run_df.sort_values("intervals").iterrows():
        x, y = load_field(row["output_dir"], field)
        if field in ("n", "I"):
            plt.semilogy(x, y, label=f'{int(row["intervals"])} intervals')
        else:
            plt.plot(x, y, label=f'{int(row["intervals"])} intervals')
    plt.xlabel("Position (nm)")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_ROOT, f"grid_independence_{field}.pdf")
    plt.savefig(plot_path, bbox_inches="tight")
    plt.show()
    print("Saved:", plot_path)

## Result

Relative to the 1000-interval solution, the 500-interval grid used in this work differs by about 7e-5 in psi, 1e-4 in n and 8e-4 in I (relative L2). These differences are an order of magnitude smaller than the PINN-versus-FD errors (about 5e-3), so the 500-interval FD solution is used as the reference.